# Sampling from a target distribution

This notebook introduces three closely related ideas:

1. **Direct sampling** from a known probability density using `scipy.stats`,
2. **Rejection sampling** when direct sampling is less convenient,
3. **Reweighting / importance sampling** to reconstruct a target distribution from samples drawn from another one.

We use a simple one-dimensional example because it is easy to visualize.
The target distribution $P(x)$ is a **mixture of two Gaussians**, while the auxiliary distribution $P'(x)$ is a **single broad Gaussian**.

The final part of the notebook shows how these sampled points can be turned back into a continuous approximation of the underlying distribution using **kernel density estimation (KDE)**.

In [ ]:
%matplotlib inline

import random
import math
import os

import numpy as np
import scipy.stats as st
from scipy.integrate import quad
import matplotlib.pyplot as plt
from IPython.display import display, Image

## Sketch from the lecture

In [ ]:
if os.path.exists("kernel.png"):
    display(Image("kernel.png"))
else:
    print("kernel.png not found in the current working directory.")

## Step 1 — Define the model distributions

We define:

- $P(x)$: the **target** probability density, taken as a sum of two Gaussians,
- $P'(x)$: a simpler **reference** density, here a single broad Gaussian.

This is a common setting in sampling:
sometimes we want to estimate properties of a difficult distribution $P(x)$
but it is easier to generate points from another distribution $P'(x)$.

In [ ]:
def gaussian(x, mu, sigma):
    """Return the value of a normalized 1D Gaussian distribution."""
    prefactor = 1.0 / (np.sqrt(2 * np.pi) * sigma)
    exponent = -0.5 * ((x - mu) / sigma) ** 2
    return prefactor * np.exp(exponent)


def pdf(x):
    """Target distribution P(x): a mixture of two Gaussians."""
    return 0.3 * gaussian(x, -3, 1) + 0.7 * gaussian(x, 3, 1)


def pdf0(x):
    """Reference distribution P'(x): a single broader Gaussian."""
    return gaussian(x, 0, 3)

## Quick checks

Before sampling, it is good practice to inspect the two densities and verify that they integrate to approximately 1 over a sufficiently wide interval.

In [ ]:
print("Integral of P(x) over [-30, 30]: ", quad(pdf, -30, 30)[0])
print("Integral of P'(x) over [-30, 30]:", quad(pdf0, -30, 30)[0])

In [ ]:
x = np.linspace(-9, 9, 400)

plt.figure(figsize=(8, 4))
plt.plot(x, pdf(x), label="P(x): target distribution", lw=2)
plt.plot(x, pdf0(x), label="P'(x): reference distribution", lw=2)
plt.xlabel("x")
plt.ylabel("Probability density")
plt.title("Target and reference distributions")
plt.legend()
plt.show()

## Step 2 — Rejection sampling

Rejection sampling works as follows:

1. Propose a point $x$ from a simple distribution (here uniform in $[x_{\min}, x_{\max}]$),
2. Draw a random height $u$,
3. Accept the point if $u < P(x)$.

Geometrically, this means we sample uniformly in a rectangle that contains the graph of $P(x)$,
and keep only the points that fall below the curve.

We estimate the maximum of the target density numerically on a dense grid.

In [ ]:
def rejection_sampling(n_samples, xmin, xmax, ngrid=5000):
    """Generate samples from P(x) using rejection sampling.

    Parameters
    ----------
    n_samples : int
        Number of accepted samples to generate.
    xmin, xmax : float
        Range used for the uniform proposal distribution.
    ngrid : int
        Number of grid points used to estimate the maximum of P(x).

    Returns
    -------
    np.ndarray
        Accepted samples distributed according to P(x).
    """
    x_grid = np.linspace(xmin, xmax, ngrid)
    proposal_max = np.max(pdf(x_grid))

    samples = []
    while len(samples) < n_samples:
        # Propose x uniformly in the chosen interval.
        x_trial = random.uniform(xmin, xmax)

        # Draw a random height in the bounding rectangle.
        u = random.uniform(0.0, proposal_max)

        # Accept only if the point lies below the target density.
        if u < pdf(x_trial):
            samples.append(x_trial)

    return np.asarray(samples)

## Step 3 — Sampling with `scipy.stats`

If a probability density is available in normalized form, `scipy.stats.rv_continuous`
can be subclassed to obtain an object with methods such as `.rvs()`.

This gives us a second route to sampling:
instead of rejection sampling, we let SciPy draw random numbers from the chosen density.

In [ ]:
npoints = 2000
xmin = -12
xmax = 12


class MyPDF(st.rv_continuous):
    def _pdf(self, x):
        return pdf(x)


class MyPDF0(st.rv_continuous):
    def _pdf(self, x):
        return pdf0(x)


my_cv = MyPDF(a=xmin, b=xmax, name="my_pdf")
my_cv0 = MyPDF0(a=xmin, b=xmax, name="my_pdf0")

In [ ]:
def random_samples(dist, npoints):
    """Draw exactly npoints samples from a scipy.stats distribution object."""
    return np.asarray([dist.rvs() for _ in range(npoints)])

Now we generate three sets of samples:

- `samples`: direct samples from the target distribution `P(x)`,
- `samples0`: samples from the reference distribution `P'(x)`,
- `rej_samples`: samples from `P(x)` produced by rejection sampling.

These sets should not look identical point-by-point, but the two sets drawn from `P(x)` should reproduce the same overall density.

In [ ]:
samples = my_cv.rvs(size=npoints)
samples0 = my_cv0.rvs(size=npoints)
rej_samples = rejection_sampling(npoints, xmin, xmax)
x_values = np.linspace(xmin, xmax, 500)

In [ ]:
# Plot the individual sampled points along the x-axis.
y = np.zeros(len(samples))
y1 = y + 0.08

plt.figure(figsize=(10, 4))
plt.scatter(samples, y, c="tab:blue", s=12, alpha=0.5, label="Samples from P(x)")
plt.scatter(samples0, y1, c="tab:red", s=12, alpha=0.5, label="Samples from P'(x)")
plt.plot(x_values, pdf(x_values), c="tab:blue", lw=2)
plt.plot(x_values, pdf0(x_values), c="tab:red", lw=2)
plt.yticks([])
plt.xlabel("x")
plt.title("Sampled points and their underlying probability densities")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(samples, bins=50, density=True, alpha=0.5, label="Direct samples from P(x)")
plt.hist(rej_samples, bins=50, density=True, alpha=0.5, label="Rejection samples from P(x)")
plt.plot(x_values, pdf(x_values), c="black", lw=2, label="Exact P(x)")
plt.xlabel("Sample value")
plt.ylabel("Density")
plt.title("Two different ways to sample the same target distribution")
plt.legend()
plt.show()

## Step 4 — Reconstruct the distribution with KDE

A histogram is simple and intuitive, but somewhat coarse.
A smoother reconstruction can be obtained with **kernel density estimation (KDE)**.

Here we first perform KDE on:

- samples drawn from $P(x)$,
- samples drawn from $P'(x)$.

This illustrates an obvious but important point:
if we estimate the density from points generated from the *wrong* distribution,
the KDE reconstructs that wrong distribution.

In [ ]:
kernel = st.gaussian_kde(samples)
kernel0 = st.gaussian_kde(samples0)

plt.figure(figsize=(10, 5))
plt.plot(x_values, kernel(x_values), c="tab:blue", lw=3, label="KDE from samples ~ P(x)")
plt.plot(x_values, kernel0(x_values), c="tab:red", lw=3, label="KDE from samples ~ P'(x)")
plt.plot(x_values, pdf(x_values), c="tab:blue", lw=1.5, ls="--", label="Exact P(x)")
plt.plot(x_values, pdf0(x_values), c="tab:red", lw=1.5, ls="--", label="Exact P'(x)")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Unweighted KDE")
plt.legend()
plt.show()

In [ ]:
print("Integral of KDE built from samples ~ P(x):", quad(kernel, xmin, xmax)[0])
print("Integral from numerical trapezoidal rule: ", np.trapz(kernel(x_values), x_values))

## Step 5 — Recover $P(x)$ from samples drawn from $P'(x)$

Suppose we only have samples from the reference distribution $P'(x)$,
but we want to reconstruct the target distribution $P(x)$.

A standard trick is **importance weighting**:
each point $x_i$ sampled from $P'(x)$ receives the weight

\[
w_i = \frac{P(x_i)}{P'(x_i)}.
\]

This compensates for the fact that the points were not drawn from the target distribution directly.

In [ ]:
weights = np.ones(len(samples0))
for i, x_i in enumerate(samples0):
    weights[i] = pdf(x_i) / pdf0(x_i)

kernel_w = st.gaussian_kde(samples0, weights=weights)

plt.figure(figsize=(10, 5))
plt.plot(x_values, kernel(x_values), c="tab:blue", lw=3, label="KDE from direct samples ~ P(x)")
plt.plot(x_values, kernel_w(x_values), c="tab:red", lw=3, label="Weighted KDE from samples ~ P'(x)")
plt.plot(x_values, pdf(x_values), c="black", lw=1.5, ls="--", label="Exact P(x)")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Recovering P(x) by reweighting samples from P'(x)")
plt.legend()
plt.show()

## Step 6 — Build KDE manually

To make the KDE formula more transparent, we now implement it explicitly.

For a set of samples \(x_i\), the KDE at position \(x\) is

\[
\hat{p}(x) = \frac{1}{n} \sum_i K_h(x - x_i),
\]

where \(K_h\) is a Gaussian kernel with bandwidth \(h\).

This cell is pedagogical: `scipy.stats.gaussian_kde` is more optimized and should usually be preferred in practice.

In [ ]:
def gaussian_kernel(x, bandwidth):
    """Gaussian kernel centered at zero with width `bandwidth`."""
    return (1.0 / (np.sqrt(2 * np.pi) * bandwidth)) * np.exp(-0.5 * (x / bandwidth) ** 2)


def kde(x, data, bandwidth):
    """Compute the 1D kernel density estimate at a single point x."""
    n = len(data)
    return np.sum([gaussian_kernel(x - xi, bandwidth) for xi in data]) / n


# Standard bandwidth estimates for univariate data.
std_dev = samples.std()
silverman_bandwidth = 1.06 * std_dev * npoints ** (-1 / 5)
scotts_bandwidth = std_dev * npoints ** (-1 / 5)

print("Silverman bandwidth:", silverman_bandwidth)
print("Scott bandwidth:    ", scotts_bandwidth)


kde_values = [kde(x, samples, scotts_bandwidth) for x in x_values]

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x_values, np.asarray(kde_values), c="tab:blue", lw=3, label="Manual KDE from samples ~ P(x)")
plt.plot(x_values, pdf(x_values), c="black", lw=1.5, ls="--", label="Exact P(x)")
plt.xlabel("x")
plt.ylabel("Density")
plt.title("Manual Gaussian KDE")
plt.legend()
plt.show()

In [ ]:
print("Integral of the Gaussian kernel over the chosen finite interval:",
      quad(gaussian_kernel, xmin, xmax, args=(scotts_bandwidth,))[0])

print("Integral of the manual KDE on the plotting grid:",
      np.trapz(np.asarray(kde_values), x_values))

## Take-home messages

- **Direct sampling** and **rejection sampling** can both reproduce the same target distribution.
- A KDE reconstructs the density associated with the points it receives.
- If the points come from a different distribution, **importance weights** can correct the estimate.
- Implementing KDE manually helps connect the formula to the smoother result produced by `gaussian_kde`.

This small 1D example captures ideas that are widely used in more advanced settings,
including statistical physics, Monte Carlo methods, and atomistic simulations.